In [ ]:
# -*- coding: utf-8 -*-
"""
ANÁLISE POR FAIXAS + PARK + AUTOENCODER + CALIBRAÇÃO SAUDÁVEL ROBUSTA

Versão feita para corrigir o problema observado:
o estado SEM DANO estava ficando ruim em RMSD/CCDM depois da compensação.

Diagnóstico do problema anterior:
- A rede/regressor estava tentando prever parâmetros térmicos pelo latente do AE.
- Isso melhorava um pouco alguns casos, mas não garantia que a curva saudável
  em cada temperatura fosse realmente levada para a referência.
- Resultado: Dano 0 ainda ficava ruim.

Nova estratégia:
1) O Autoencoder continua no código e continua sendo treinado apenas com curvas saudáveis.
   Ele serve como diagnóstico térmico/latente da pesquisa.
2) A compensação final NÃO depende mais de parâmetros previstos pelo latente.
3) A compensação usa uma calibração saudável robusta por temperatura medida:

       H_T  = curva saudável mediana/interpolada na temperatura T
       H_REF = curva saudável de referência

   Aprende, usando apenas H_T e H_REF, uma transformação física simples:

       X_comp(f) = a * X(f + tau) + b + c*z(f)

   onde:
       tau = deslocamento horizontal estimado pelos fundos suavizados saudáveis
       a   = ganho vertical
       b   = offset vertical
       c   = inclinação linear
       z   = eixo normalizado de frequência

4) A mesma transformação saudável é aplicada à curva original X, inclusive se tiver dano.

Por que isso melhora o sem dano:
- Se X for saudável, X ≈ H_T.
- Então a transformação foi calculada exatamente para levar H_T para H_REF.
- Logo Dano 0 tende a ter RMSD/CCDM muito melhores.

Por que ainda preserva dano:
- A rede não gera a curva.
- A referência bruta não é copiada ponto a ponto.
- A transformação tem só poucos graus de liberdade.
- Os picos locais da curva original continuam vindo da curva original.

Saídas:
- df_long_todas_faixas.csv
- resumo_por_faixa_metodo_dano.csv
- ranking_faixas.csv
- monotonicidade_por_faixa.csv
- heatmaps PNG/PDF
"""

# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pandas.errors import PerformanceWarning

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=PerformanceWarning)

np.random.seed(42)
torch.manual_seed(42)


# ============================================================
# 2) PARÂMETROS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30

OUTPUT_DIR = "ANALISE_FAIXAS_AE_CALIBRACAO_SAUDAVEL"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# FAIXAS DE FREQUÊNCIA
# ------------------------------------------------------------
# Para reproduzir algo parecido com o heatmap que você mostrou:
# 30–40, 35–45, 40–50, ..., 90–100 kHz
#
# Se quiser de 20 em 20 kHz:
# FREQ_START_KHZ = 20
# FREQ_END_KHZ   = 100
# BAND_WIDTH_KHZ = 20
# BAND_STEP_KHZ  = 20

FREQ_START_KHZ = 30
FREQ_END_KHZ = 100
BAND_WIDTH_KHZ = 10
BAND_STEP_KHZ = 5

MIN_POINTS_PER_BAND = 50

# ------------------------------------------------------------
# MÉTODOS
# ------------------------------------------------------------

RUN_ORIGINAL = True
RUN_PARK = True
RUN_AE_CALIBRADO = True

# O AE agora é diagnóstico/latente.
# Se quiser acelerar MUITO, coloque TRAIN_AE_DIAGNOSTIC = False.
TRAIN_AE_DIAGNOSTIC = True

# ------------------------------------------------------------
# AUTOENCODER
# ------------------------------------------------------------

EPOCHS = 350
BATCH_SIZE = 8
LR = 4e-4
PATIENCE = 70

LATENT_DIM = 10

INPUT_NOISE_STD = 0.02
DROPOUT = 0.15

LAMBDA_RECON = 1.00
LAMBDA_DERIV = 0.30
LAMBDA_CURVATURE = 0.12
LAMBDA_TEMP_AUX = 0.35

NUM_WORKERS = 0

# ------------------------------------------------------------
# CALIBRAÇÃO SAUDÁVEL ROBUSTA
# ------------------------------------------------------------

# Deslocamento horizontal térmico estimado em H_T -> H_REF.
USE_HEALTHY_SHIFT = True
SHIFT_MAX_FRAC = 0.035
SHIFT_NSTEPS = 181

# Transformação vertical/tendência.
USE_GAIN = True
USE_OFFSET = True
USE_TILT = True

# Limites físicos. Se D0 continuar ruim, aumente um pouco OFFSET/TILT.
GAIN_MIN = 0.70
GAIN_MAX = 1.40

OFFSET_FRAC_LIMIT = 1.20
TILT_FRAC_LIMIT = 0.80

# Alpha da transformação.
# 1.00 = aplica toda a calibração saudável.
# Se estiver deformando D1/D2, teste 0.85.
ALPHA_CALIB = 1.00

# Janela de suavização usada APENAS para ajustar tau/a/b/c.
# A curva final continua usando a curva bruta original.
FIT_SMOOTH_WIN = 401

# Peso anti-pico no ajuste dos parâmetros.
# Reduz influência de picos muito estreitos na estimação de a/b/c/tau.
USE_ROBUST_WEIGHTS = True
CURVATURE_WEIGHT_POWER = 1.5
MIN_WEIGHT = 0.10

# Interpolar saudável entre temperaturas disponíveis.
USE_TEMP_INTERPOLATION = True

# ------------------------------------------------------------
# PARK
# ------------------------------------------------------------

PARK_MAX_SHIFT_FRAC = 0.10
PARK_NSTEPS = 101
PARK_SMOOTH_WIN = 1

# ------------------------------------------------------------
# GRÁFICOS
# ------------------------------------------------------------

SALVAR_GRAFICOS_RESUMO = True

PLOTAR_CURVAS_EXEMPLO = False
TEMP_EXEMPLO = 55
DANOS_PLOTAR = [0, 1, 2]
OCORRENCIA_CURVA = 0


# ============================================================
# 3) FUNÇÕES GERAIS
# ============================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_all_freq_columns(df):
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)

        if f is not None:
            cols.append(c)
            freqs.append(f)

    if len(cols) == 0:
        raise ValueError("Nenhuma coluna de frequência f_...Hz foi encontrada.")

    order = np.argsort(freqs)

    fcols = [cols[i] for i in order]
    fhz = np.array(freqs, dtype=float)[order]

    return fcols, fhz


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)

        if f is not None:
            f_khz = f / 1e3

            if fmin_khz <= f_khz < fmax_khz:
                cols.append(c)
                freqs.append(f)

    if len(cols) == 0:
        return [], np.array([], dtype=float)

    order = np.argsort(freqs)

    fcols = [cols[i] for i in order]
    fhz = np.array(freqs, dtype=float)[order]

    return fcols, fhz


def gerar_faixas(df):
    _, all_fhz = get_all_freq_columns(df)

    fmin_data = float(np.min(all_fhz) / 1e3)
    fmax_data = float(np.max(all_fhz) / 1e3)

    start = float(FREQ_START_KHZ)
    end = float(FREQ_END_KHZ)

    faixas = []

    f0 = start
    while f0 + BAND_WIDTH_KHZ <= end + 1e-12:
        f1 = f0 + BAND_WIDTH_KHZ

        if f1 >= fmin_data and f0 <= fmax_data:
            faixas.append((float(f0), float(f1)))

        f0 += BAND_STEP_KHZ

    return faixas


def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 18,
        "axes.labelsize": 20,
        "axes.titlesize": 22,
        "xtick.labelsize": 16,
        "ytick.labelsize": 18,
        "legend.fontsize": 13,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


def salvar_figura(fig, nome_base, output_dir=OUTPUT_DIR, dpi=500):
    os.makedirs(output_dir, exist_ok=True)

    png_path = os.path.join(output_dir, f"{nome_base}.png")
    pdf_path = os.path.join(output_dir, f"{nome_base}.pdf")

    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")

    print(f"Figura salva em PNG: {png_path}")
    print(f"Figura salva em PDF: {pdf_path}")

    return png_path, pdf_path


def moving_average(arr, win):
    arr = np.asarray(arr, dtype=float)

    if win <= 1:
        return arr.copy()

    if win % 2 == 0:
        win += 1

    if len(arr) < win:
        win = max(3, len(arr) // 5)
        if win % 2 == 0:
            win += 1

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win

    return np.convolve(arr_pad, kernel, mode="valid")


def smooth_fit_curve(x):
    return moving_average(x, FIT_SMOOTH_WIN)


def shift_interp(x, fHz, tau):
    f_shift = fHz + tau

    return np.interp(
        fHz,
        f_shift,
        x,
        left=x[0],
        right=x[-1]
    )


def formatar_faixa(f0, f1):
    return f"{int(f0)}–{int(f1)} kHz"


def nome_faixa(f0, f1):
    return f"{int(f0)}_{int(f1)}kHz"


# ============================================================
# 4) REFERÊNCIAS SAUDÁVEIS
# ============================================================

def get_healthy_references_by_temperature(df, fcols, ref_temp):
    df_h = df[df["falha"] == 0].copy()

    if len(df_h) == 0:
        raise ValueError("Não há curvas saudáveis, isto é, falha = 0.")

    healthy_by_temp = {}
    temps_h = np.array(sorted(df_h["temperatura_c"].unique()), dtype=float)

    for T in temps_h:
        pool = df_h.loc[
            np.isclose(df_h["temperatura_c"], T),
            fcols
        ].to_numpy(np.float32)

        healthy_by_temp[float(T)] = np.median(pool, axis=0)

    ref_temp_used = float(temps_h[np.argmin(np.abs(temps_h - ref_temp))])
    y_ref_healthy = healthy_by_temp[ref_temp_used]

    return healthy_by_temp, temps_h, y_ref_healthy, ref_temp_used


def get_nearest_healthy_curve(healthy_by_temp, healthy_temps, temperatura):
    healthy_temps = np.asarray(healthy_temps, dtype=float)

    temp_used = float(
        healthy_temps[np.argmin(np.abs(healthy_temps - temperatura))]
    )

    return healthy_by_temp[temp_used], temp_used


def interpolate_healthy_curve(healthy_by_temp, healthy_temps, temperatura):
    healthy_temps = np.asarray(healthy_temps, dtype=float)
    T = float(temperatura)

    if not USE_TEMP_INTERPOLATION:
        return get_nearest_healthy_curve(
            healthy_by_temp,
            healthy_temps,
            T
        )

    if T <= healthy_temps[0]:
        return healthy_by_temp[float(healthy_temps[0])].copy(), float(healthy_temps[0])

    if T >= healthy_temps[-1]:
        return healthy_by_temp[float(healthy_temps[-1])].copy(), float(healthy_temps[-1])

    j = np.searchsorted(healthy_temps, T)

    T0 = float(healthy_temps[j - 1])
    T1 = float(healthy_temps[j])

    h0 = healthy_by_temp[T0]
    h1 = healthy_by_temp[T1]

    w = (T - T0) / (T1 - T0 + 1e-12)

    h = (1.0 - w) * h0 + w * h1

    return h, T


# ============================================================
# 5) MÉTRICAS
# ============================================================

def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2)) + 1e-18)

    corr = num / den

    return float(1 - corr)


def derivative_loss(y_pred, y_true):
    dy_pred = y_pred[:, 1:] - y_pred[:, :-1]
    dy_true = y_true[:, 1:] - y_true[:, :-1]

    return F.smooth_l1_loss(dy_pred, dy_true)


def curvature_loss(y_pred, y_true):
    d2_pred = y_pred[:, 2:] - 2 * y_pred[:, 1:-1] + y_pred[:, :-2]
    d2_true = y_true[:, 2:] - 2 * y_true[:, 1:-1] + y_true[:, :-2]

    return F.smooth_l1_loss(d2_pred, d2_true)


def calcular_metricas_com_preservacao(
    df_curvas,
    df_original,
    fcols,
    y_ref_healthy,
    healthy_by_temp,
    healthy_temps,
    metodo,
    f0,
    f1
):
    X_comp = df_curvas[fcols].to_numpy(np.float32)
    X_orig = df_original[fcols].to_numpy(np.float32)

    meta_cols = [c for c in df_curvas.columns if c not in fcols]
    df_out = df_curvas[meta_cols].copy()

    rmsd_list = []
    ccdm_list = []
    damage_res_rmsd = []
    damage_res_ccdm = []
    alteracao_rmsd = []

    for i, (_, row) in enumerate(df_original.iterrows()):
        T = float(row["temperatura_c"])

        h_T, _ = get_nearest_healthy_curve(
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            temperatura=T
        )

        y_comp = X_comp[i]
        y_orig = X_orig[i]

        rmsd_list.append(rmsd(y_comp, y_ref_healthy))
        ccdm_list.append(ccdm(y_comp, y_ref_healthy))

        assinatura_esperada = y_orig - h_T
        assinatura_saida = y_comp - y_ref_healthy

        damage_res_rmsd.append(rmsd(assinatura_saida, assinatura_esperada))
        damage_res_ccdm.append(ccdm(assinatura_saida, assinatura_esperada))

        alteracao_rmsd.append(rmsd(y_comp, y_orig))

    df_out["RMSD"] = rmsd_list
    df_out["CCDM"] = ccdm_list
    df_out["DamageResidual_RMSD"] = damage_res_rmsd
    df_out["DamageResidual_CCDM"] = damage_res_ccdm
    df_out["Alteracao_RMSD"] = alteracao_rmsd
    df_out["Metodo"] = metodo
    df_out["Freq_min_kHz"] = f0
    df_out["Freq_max_kHz"] = f1
    df_out["Faixa"] = formatar_faixa(f0, f1)

    return df_out


# ============================================================
# 6) AUTOENCODER SAUDÁVEL PARA DIAGNÓSTICO
# ============================================================

class HealthyThermalAE(nn.Module):
    def __init__(self, n_points, latent_dim=10, dropout=0.15):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(n_points, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),

            nn.Linear(64, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),

            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.GELU(),

            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.GELU(),

            nn.Linear(256, n_points)
        )

        self.temp_head = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        t = self.temp_head(z)

        return y, z, t


def treinar_autoencoder_saudavel(df, fcols):
    df_h = df[df["falha"] == 0].copy()

    Xh = df_h[fcols].to_numpy(np.float32)
    T_h = df_h["temperatura_c"].to_numpy(np.float32)

    scaler = StandardScaler()
    Xh_s = scaler.fit_transform(Xh).astype(np.float32)

    T_aux = ((T_h - REF_TEMP) / 100.0).reshape(-1, 1).astype(np.float32)

    indices = np.arange(len(Xh_s))

    if len(indices) >= 10:
        idx_train, idx_val = train_test_split(
            indices,
            test_size=0.20,
            random_state=42
        )
    else:
        idx_train = indices
        idx_val = indices

    X_tensor = torch.tensor(Xh_s, dtype=torch.float32)
    T_tensor = torch.tensor(T_aux, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_tensor[idx_train], T_tensor[idx_train]),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        TensorDataset(X_tensor[idx_val], T_tensor[idx_val]),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = HealthyThermalAE(
        n_points=Xh_s.shape[1],
        latent_dim=LATENT_DIM,
        dropout=DROPOUT
    ).to(device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=0.5,
        patience=30
    )

    huber = nn.SmoothL1Loss()
    mse = nn.MSELoss()

    best_val = np.inf
    best_state = copy.deepcopy(model.state_dict())
    epochs_sem_melhora = 0

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_recon": [],
        "val_temp": [],
        "lr": []
    }

    for ep in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for xb, tb in train_loader:
            xb = xb.to(device)
            tb = tb.to(device)

            if INPUT_NOISE_STD > 0:
                xb_in = xb + INPUT_NOISE_STD * torch.randn_like(xb)
            else:
                xb_in = xb

            opt.zero_grad()

            pred, z, tpred = model(xb_in)

            loss_recon = huber(pred, xb)
            loss_deriv = derivative_loss(pred, xb)
            loss_curv = curvature_loss(pred, xb)
            loss_temp = mse(tpred, tb)

            loss = (
                LAMBDA_RECON * loss_recon
                + LAMBDA_DERIV * loss_deriv
                + LAMBDA_CURVATURE * loss_curv
                + LAMBDA_TEMP_AUX * loss_temp
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            opt.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        val_recons = []
        val_temps = []

        with torch.no_grad():
            for xb, tb in val_loader:
                xb = xb.to(device)
                tb = tb.to(device)

                pred, z, tpred = model(xb)

                loss_recon = huber(pred, xb)
                loss_deriv = derivative_loss(pred, xb)
                loss_curv = curvature_loss(pred, xb)
                loss_temp = mse(tpred, tb)

                loss = (
                    LAMBDA_RECON * loss_recon
                    + LAMBDA_DERIV * loss_deriv
                    + LAMBDA_CURVATURE * loss_curv
                    + LAMBDA_TEMP_AUX * loss_temp
                )

                val_losses.append(loss.item())
                val_recons.append(loss_recon.item())
                val_temps.append(loss_temp.item())

        train_mean = float(np.mean(train_losses))
        val_mean = float(np.mean(val_losses))
        val_recon = float(np.mean(val_recons))
        val_temp = float(np.mean(val_temps))

        scheduler.step(val_mean)

        current_lr = opt.param_groups[0]["lr"]

        history["epoch"].append(ep)
        history["train_loss"].append(train_mean)
        history["val_loss"].append(val_mean)
        history["val_recon"].append(val_recon)
        history["val_temp"].append(val_temp)
        history["lr"].append(current_lr)

        if val_mean < best_val - 1e-7:
            best_val = val_mean
            best_state = copy.deepcopy(model.state_dict())
            epochs_sem_melhora = 0
        else:
            epochs_sem_melhora += 1

        if ep == 1 or ep % 50 == 0:
            print(
                f"Epoch {ep:4d}/{EPOCHS} | "
                f"train={train_mean:.6f} | "
                f"val={val_mean:.6f} | "
                f"recon={val_recon:.6f} | "
                f"temp={val_temp:.6f} | "
                f"lr={current_lr:.2e}"
            )

        if epochs_sem_melhora >= PATIENCE:
            print(f"Early stopping na epoch {ep}. Melhor val_loss = {best_val:.6f}")
            break

    model.load_state_dict(best_state)

    return {
        "model": model,
        "scaler": scaler,
        "device": device,
        "history": pd.DataFrame(history)
    }


def aplicar_ae_todas_curvas(df, fcols, ae_extra):
    model = ae_extra["model"]
    scaler = ae_extra["scaler"]
    device = ae_extra["device"]

    X = df[fcols].to_numpy(np.float32)
    Xs = scaler.transform(X).astype(np.float32)

    X_tensor = torch.tensor(Xs, dtype=torch.float32)

    model.eval()

    Z_list = []
    Recon_list = []
    Tpred_list = []

    with torch.no_grad():
        for i in range(0, len(X_tensor), BATCH_SIZE):
            xb = X_tensor[i:i + BATCH_SIZE].to(device)

            pred, z, tpred = model(xb)

            Z_list.append(z.cpu().numpy())
            Recon_list.append(pred.cpu().numpy())
            Tpred_list.append(tpred.cpu().numpy())

    Z = np.vstack(Z_list).astype(np.float32, copy=False)
    Recon_s = np.vstack(Recon_list).astype(np.float32, copy=False)
    Recon = scaler.inverse_transform(Recon_s).astype(np.float32, copy=False)

    T_pred = np.vstack(Tpred_list)[:, 0] * 100.0 + REF_TEMP
    T_pred = T_pred.astype(np.float32, copy=False)

    return Z, Recon, T_pred


# ============================================================
# 7) CALIBRAÇÃO SAUDÁVEL ROBUSTA
# ============================================================

def calcular_pesos_robustos(x_fit, ref_fit):
    if not USE_ROBUST_WEIGHTS:
        return np.ones_like(ref_fit, dtype=float)

    d2_x = np.abs(np.gradient(np.gradient(x_fit)))
    d2_ref = np.abs(np.gradient(np.gradient(ref_fit)))

    curv = d2_x + d2_ref
    scale = np.median(curv) + 1e-12

    score = curv / scale
    w = 1.0 / (1.0 + score ** CURVATURE_WEIGHT_POWER)
    w = np.clip(w, MIN_WEIGHT, 1.0)

    return w.astype(float)


def estimar_tau_saudavel(h_T, h_ref, fHz):
    if not USE_HEALTHY_SHIFT:
        return 0.0

    df_band = fHz[-1] - fHz[0]
    tau_max = SHIFT_MAX_FRAC * df_band

    taus = np.linspace(-tau_max, tau_max, SHIFT_NSTEPS)

    hT_fit = smooth_fit_curve(h_T)
    ref_fit = smooth_fit_curve(h_ref)

    best_tau = 0.0
    best_err = np.inf

    for tau in taus:
        hT_shift = shift_interp(hT_fit, fHz, tau)

        w = calcular_pesos_robustos(hT_shift, ref_fit)

        err = np.average((hT_shift - ref_fit) ** 2, weights=w)

        if err < best_err:
            best_err = err
            best_tau = tau

    return float(best_tau)


def ajustar_affine_saudavel(h_T_shift, h_ref):
    """
    Ajusta:
        h_ref ~= a*h_T_shift + b + c*z

    usando curvas suavizadas e pesos anti-pico.
    """

    x_fit = smooth_fit_curve(h_T_shift)
    ref_fit = smooth_fit_curve(h_ref)

    z = np.linspace(-1.0, 1.0, len(h_T_shift))

    cols = []

    if USE_GAIN:
        cols.append(x_fit)
    else:
        cols.append(np.ones_like(x_fit))

    if USE_OFFSET:
        cols.append(np.ones_like(x_fit))

    if USE_TILT:
        cols.append(z)

    A = np.column_stack(cols)

    w = calcular_pesos_robustos(x_fit, ref_fit)
    sw = np.sqrt(w)

    A_w = A * sw[:, None]
    b_w = ref_fit * sw

    coef, *_ = np.linalg.lstsq(A_w, b_w, rcond=None)

    # Interpreta coeficientes.
    pos = 0

    if USE_GAIN:
        a = coef[pos]
        pos += 1
    else:
        a = 1.0
        pos += 1  # coeficiente da coluna ones ignorado abaixo se sem gain

    if USE_OFFSET:
        b = coef[pos]
        pos += 1
    else:
        b = 0.0

    if USE_TILT:
        c = coef[pos]
    else:
        c = 0.0

    ref_amp = np.ptp(h_ref) + 1e-12

    a = float(np.clip(a, GAIN_MIN, GAIN_MAX))
    b = float(np.clip(b, -OFFSET_FRAC_LIMIT * ref_amp, OFFSET_FRAC_LIMIT * ref_amp))
    c = float(np.clip(c, -TILT_FRAC_LIMIT * ref_amp, TILT_FRAC_LIMIT * ref_amp))

    return a, b, c


def compensar_ae_calibracao_saudavel(
    df,
    fcols,
    fhz,
    healthy_by_temp,
    healthy_temps,
    y_ref_healthy,
    ae_extra=None
):
    """
    Compensação principal desta versão.

    O AE pode existir para diagnóstico, mas a compensação usa calibração saudável
    por temperatura medida, porque isso força Dano 0 a ser bem compensado.
    """

    X = df[fcols].to_numpy(np.float32)

    meta = df.drop(columns=fcols).copy()

    if ae_extra is not None:
        Z_all, Recon_all, T_pred = aplicar_ae_todas_curvas(df, fcols, ae_extra)
    else:
        Z_all = None
        Recon_all = np.zeros_like(X, dtype=np.float32)
        T_pred = np.full(len(df), np.nan, dtype=np.float32)

    Y_comp = np.zeros_like(X, dtype=np.float32)

    tau_list = []
    a_list = []
    b_list = []
    c_list = []

    for i, (_, row) in enumerate(df.iterrows()):
        T = float(row["temperatura_c"])

        h_T, T_used = interpolate_healthy_curve(
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            temperatura=T
        )

        tau = estimar_tau_saudavel(
            h_T=h_T,
            h_ref=y_ref_healthy,
            fHz=fhz
        )

        h_T_shift = shift_interp(h_T, fhz, tau)

        a, b, c = ajustar_affine_saudavel(
            h_T_shift=h_T_shift,
            h_ref=y_ref_healthy
        )

        x_shift = shift_interp(X[i], fhz, tau)

        z = np.linspace(-1.0, 1.0, len(x_shift))

        y_full = a * x_shift + b + c * z

        # Alpha permite suavizar a compensação se necessário.
        y_comp = X[i] + ALPHA_CALIB * (y_full - X[i])

        Y_comp[i] = y_comp.astype(np.float32)

        tau_list.append(tau)
        a_list.append(a)
        b_list.append(b)
        c_list.append(c)

    df_comp = pd.concat(
        [
            meta,
            pd.DataFrame(Y_comp, columns=fcols, index=df.index)
        ],
        axis=1
    )

    df_comp["tau_calib"] = tau_list
    df_comp["gain_calib"] = a_list
    df_comp["offset_calib"] = b_list
    df_comp["tilt_calib"] = c_list
    df_comp["temperatura_ae_pred"] = T_pred

    df_recon = pd.concat(
        [
            meta.copy(),
            pd.DataFrame(Recon_all, columns=fcols, index=df.index)
        ],
        axis=1
    )

    return df_comp, df_recon, Z_all, T_pred


# ============================================================
# 8) PARK
# ============================================================

def park_single(x, ref, fHz):
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    taus = np.linspace(-tau_max, tau_max, PARK_NSTEPS)

    for tau in taus:
        x_shift = shift_interp(x, fHz, tau)

        dS = np.mean(ref - x_shift)

        y_try = x_shift + dS

        err = np.mean((ref - y_try) ** 2)

        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y_comp = shift_interp(x, fHz, best_tau) + best_dS
    y_comp = moving_average(y_comp, PARK_SMOOTH_WIN)

    return y_comp


def compensar_park(df, fcols, fHz, y_ref_healthy):
    X_all = df[fcols].to_numpy(np.float32)
    Y_comp = np.zeros_like(X_all, dtype=np.float32)

    for i in range(len(X_all)):
        Y_comp[i] = park_single(
            x=X_all[i],
            ref=y_ref_healthy,
            fHz=fHz
        )

    meta = df.drop(columns=fcols).copy()

    df_comp = pd.concat(
        [
            meta,
            pd.DataFrame(Y_comp, columns=fcols, index=df.index)
        ],
        axis=1
    )

    return df_comp


# ============================================================
# 9) RESUMOS, RANKING E HEATMAPS
# ============================================================

def resumo_por_faixa(df_long):
    resumo = (
        df_long
        .groupby(["Faixa", "Freq_min_kHz", "Freq_max_kHz", "Metodo", "falha"])[[
            "RMSD",
            "CCDM",
            "DamageResidual_RMSD",
            "DamageResidual_CCDM",
            "Alteracao_RMSD"
        ]]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )

    return resumo


def checar_monotonicidade_por_faixa(df_long):
    registros = []

    for faixa in sorted(df_long["Faixa"].unique()):
        df_f = df_long[df_long["Faixa"] == faixa]

        f0 = df_f["Freq_min_kHz"].iloc[0]
        f1 = df_f["Freq_max_kHz"].iloc[0]

        for metodo in sorted(df_f["Metodo"].unique()):
            df_m = df_f[df_f["Metodo"] == metodo]

            temps = sorted(df_m["temperatura_c"].unique())

            for T in temps:
                df_t = df_m[np.isclose(df_m["temperatura_c"], T)]

                if not {0, 1, 2}.issubset(set(df_t["falha"].unique())):
                    continue

                for metrica in ["RMSD", "CCDM"]:
                    medias = {}

                    for d in [0, 1, 2]:
                        medias[d] = df_t.loc[
                            df_t["falha"] == d,
                            metrica
                        ].mean()

                    ok = medias[0] < medias[1] < medias[2]

                    registros.append({
                        "Faixa": faixa,
                        "Freq_min_kHz": f0,
                        "Freq_max_kHz": f1,
                        "Metodo": metodo,
                        "Temperatura": T,
                        "Metrica": metrica,
                        "D0": medias[0],
                        "D1": medias[1],
                        "D2": medias[2],
                        "Monotonico_D0_D1_D2": ok
                    })

    return pd.DataFrame(registros)


def criar_ranking_faixas(df_long, metodo_alvo="AE calib saudável"):
    registros = []

    df_use = df_long[df_long["Metodo"] == metodo_alvo].copy()

    for faixa in sorted(df_use["Faixa"].unique()):
        df_f = df_use[df_use["Faixa"] == faixa]

        f0 = df_f["Freq_min_kHz"].iloc[0]
        f1 = df_f["Freq_max_kHz"].iloc[0]

        medias_rmsd = {}
        medias_ccdm = {}
        medias_pres = {}

        for d in [0, 1, 2]:
            df_d = df_f[df_f["falha"] == d]

            medias_rmsd[d] = df_d["RMSD"].mean()
            medias_ccdm[d] = df_d["CCDM"].mean()
            medias_pres[d] = df_d["DamageResidual_RMSD"].mean()

        sep_10_rmsd = medias_rmsd[1] - medias_rmsd[0]
        sep_21_rmsd = medias_rmsd[2] - medias_rmsd[1]

        sep_10_ccdm = medias_ccdm[1] - medias_ccdm[0]
        sep_21_ccdm = medias_ccdm[2] - medias_ccdm[1]

        pres_mean = np.mean([medias_pres[0], medias_pres[1], medias_pres[2]])

        mono_score = 0.0

        if sep_10_rmsd > 0:
            mono_score += 0.25
        if sep_21_rmsd > 0:
            mono_score += 0.25
        if sep_10_ccdm > 0:
            mono_score += 0.25
        if sep_21_ccdm > 0:
            mono_score += 0.25

        # Agora D0 baixo pesa bastante, porque era o problema.
        score = (
            2.0 * mono_score
            + 0.20 * max(sep_10_rmsd, 0)
            + 0.20 * max(sep_21_rmsd, 0)
            + 0.50 * max(sep_10_ccdm, 0)
            + 0.50 * max(sep_21_ccdm, 0)
            - 0.12 * medias_rmsd[0]
            - 1.00 * medias_ccdm[0]
            - 0.04 * pres_mean
        )

        registros.append({
            "Faixa": faixa,
            "Freq_min_kHz": f0,
            "Freq_max_kHz": f1,
            "RMSD_D0": medias_rmsd[0],
            "RMSD_D1": medias_rmsd[1],
            "RMSD_D2": medias_rmsd[2],
            "CCDM_D0": medias_ccdm[0],
            "CCDM_D1": medias_ccdm[1],
            "CCDM_D2": medias_ccdm[2],
            "Sep_RMSD_D1_D0": sep_10_rmsd,
            "Sep_RMSD_D2_D1": sep_21_rmsd,
            "Sep_CCDM_D1_D0": sep_10_ccdm,
            "Sep_CCDM_D2_D1": sep_21_ccdm,
            "DamageResidual_RMSD_mean": pres_mean,
            "Mono_score_simples": mono_score,
            "Score_ranking": score
        })

    df_rank = pd.DataFrame(registros)

    if len(df_rank) > 0:
        df_rank = df_rank.sort_values("Score_ranking", ascending=False)

    return df_rank


def matriz_media_por_faixa_dano(df_long, metodo, metrica):
    df_m = df_long[df_long["Metodo"] == metodo].copy()

    if len(df_m) == 0:
        raise ValueError(f"Nenhum dado encontrado para Metodo = {metodo}")

    ordem_faixas = (
        df_m[["Faixa", "Freq_min_kHz", "Freq_max_kHz"]]
        .drop_duplicates()
        .sort_values("Freq_min_kHz")
    )

    faixas = ordem_faixas["Faixa"].to_list()
    danos = sorted(df_m["falha"].unique())

    matriz = np.full((len(danos), len(faixas)), np.nan, dtype=float)

    for i, d in enumerate(danos):
        for j, faixa in enumerate(faixas):
            vals = df_m.loc[
                (df_m["falha"] == d) &
                (df_m["Faixa"] == faixa),
                metrica
            ].dropna().to_numpy(float)

            if len(vals) > 0:
                matriz[i, j] = np.mean(vals)

    return matriz, danos, faixas


def formatar_valor_heatmap(v):
    if np.isnan(v):
        return ""

    if abs(v) >= 10:
        return f"{v:.1f}"

    if abs(v) >= 1:
        return f"{v:.2f}"

    return f"{v:.3g}"


def plot_heatmap_metrica(
    df_long,
    metodo,
    metrica,
    titulo=None,
    nome_arquivo=None,
    cmap="viridis",
    salvar=True,
    show=True
):
    aplicar_estilo_artigo()

    matriz, danos, faixas = matriz_media_por_faixa_dano(
        df_long=df_long,
        metodo=metodo,
        metrica=metrica
    )

    fig, ax = plt.subplots(figsize=(1.15 * len(faixas) + 4, 5.8), dpi=300)

    im = ax.imshow(
        matriz,
        aspect="auto",
        cmap=cmap
    )

    ax.set_xticks(np.arange(len(faixas)))
    ax.set_xticklabels(
        faixas,
        rotation=45,
        ha="right",
        fontsize=22
    )

    ax.set_yticks(np.arange(len(danos)))
    ax.set_yticklabels(
        [f"Dano {int(d)}" for d in danos],
        fontsize=24
    )

    ax.set_xlabel("Faixa de frequência", fontsize=30)
    ax.set_ylabel("Estado estrutural", fontsize=30)

    if titulo is None:
        titulo = f"{metrica} médio — {metodo}"

    ax.set_title(titulo, fontsize=34, pad=24)

    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            ax.text(
                j,
                i,
                formatar_valor_heatmap(matriz[i, j]),
                ha="center",
                va="center",
                color="black",
                fontsize=14
            )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metrica, fontsize=28)
    cbar.ax.tick_params(labelsize=20)

    ax.grid(False)

    plt.tight_layout()

    if nome_arquivo is None:
        nome_arquivo = (
            f"Heatmap_{metrica}_{metodo}"
            .replace(" ", "_")
            .replace("—", "_")
            .replace("/", "_")
        )

    if salvar:
        salvar_figura(fig, nome_arquivo)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, ax


def plot_heatmaps_comparacao(df_long):
    for metodo in ["Original", "Park", "AE calib saudável"]:
        if metodo not in df_long["Metodo"].unique():
            continue

        for metrica in ["RMSD", "CCDM", "DamageResidual_RMSD", "DamageResidual_CCDM"]:
            plot_heatmap_metrica(
                df_long=df_long,
                metodo=metodo,
                metrica=metrica,
                titulo=f"{metrica} médio — {metodo}",
                nome_arquivo=f"Heatmap_{metrica}_medio_{metodo}".replace(" ", "_"),
                salvar=True,
                show=True
            )


# ============================================================
# 10) EXECUÇÃO POR FAIXA
# ============================================================

def analisar_faixa(df, f0, f1):
    print("\n" + "=" * 80)
    print(f"ANALISANDO FAIXA {formatar_faixa(f0, f1)}")
    print("=" * 80)

    fcols, fhz = get_freq_columns(df, f0, f1)

    if len(fcols) < MIN_POINTS_PER_BAND:
        print(f"Faixa ignorada: apenas {len(fcols)} pontos.")
        return None

    print(f"Número de pontos na faixa: {len(fcols)}")

    healthy_by_temp, healthy_temps, y_ref_healthy, ref_temp_used = (
        get_healthy_references_by_temperature(
            df=df,
            fcols=fcols,
            ref_temp=REF_TEMP
        )
    )

    partes = []

    if RUN_ORIGINAL:
        df_original_metricas = calcular_metricas_com_preservacao(
            df_curvas=df,
            df_original=df,
            fcols=fcols,
            y_ref_healthy=y_ref_healthy,
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            metodo="Original",
            f0=f0,
            f1=f1
        )

        partes.append(df_original_metricas)

    if RUN_PARK:
        print("Aplicando Park...")
        df_park_curvas = compensar_park(
            df=df,
            fcols=fcols,
            fHz=fhz,
            y_ref_healthy=y_ref_healthy
        )

        df_park_metricas = calcular_metricas_com_preservacao(
            df_curvas=df_park_curvas,
            df_original=df,
            fcols=fcols,
            y_ref_healthy=y_ref_healthy,
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            metodo="Park",
            f0=f0,
            f1=f1
        )

        partes.append(df_park_metricas)

    if RUN_AE_CALIBRADO:
        ae_extra = None

        if TRAIN_AE_DIAGNOSTIC:
            print("Treinando AE saudável para diagnóstico/latente...")
            ae_extra = treinar_autoencoder_saudavel(
                df=df,
                fcols=fcols
            )

            pasta_faixa = os.path.join(OUTPUT_DIR, f"faixa_{nome_faixa(f0, f1)}")
            os.makedirs(pasta_faixa, exist_ok=True)

            ae_extra["history"].to_csv(
                os.path.join(pasta_faixa, "historico_loss_ae.csv"),
                index=False
            )

        print("Aplicando AE + calibração saudável robusta...")
        df_ae_curvas, df_recon_curvas, ae_latent, temp_pred_ae = (
            compensar_ae_calibracao_saudavel(
                df=df,
                fcols=fcols,
                fhz=fhz,
                healthy_by_temp=healthy_by_temp,
                healthy_temps=healthy_temps,
                y_ref_healthy=y_ref_healthy,
                ae_extra=ae_extra
            )
        )

        df_ae_metricas = calcular_metricas_com_preservacao(
            df_curvas=df_ae_curvas,
            df_original=df,
            fcols=fcols,
            y_ref_healthy=y_ref_healthy,
            healthy_by_temp=healthy_by_temp,
            healthy_temps=healthy_temps,
            metodo="AE calib saudável",
            f0=f0,
            f1=f1
        )

        partes.append(df_ae_metricas)

        pasta_faixa = os.path.join(OUTPUT_DIR, f"faixa_{nome_faixa(f0, f1)}")
        os.makedirs(pasta_faixa, exist_ok=True)

        pd.DataFrame({
            "temperatura_real": df["temperatura_c"].to_numpy(float),
            "temperatura_ae_pred": temp_pred_ae
        }).to_csv(
            os.path.join(pasta_faixa, "temperatura_ae_pred.csv"),
            index=False
        )

        try:
            if np.all(np.isfinite(temp_pred_ae)):
                r2_temp = r2_score(df["temperatura_c"].to_numpy(float), temp_pred_ae)
                mae_temp = np.mean(np.abs(df["temperatura_c"].to_numpy(float) - temp_pred_ae))
                print(f"Temperatura AE | R2={r2_temp:.4f} | MAE={mae_temp:.4f} °C")
        except Exception:
            pass

    df_faixa = pd.concat(partes, axis=0, ignore_index=True)

    pasta_faixa = os.path.join(OUTPUT_DIR, f"faixa_{nome_faixa(f0, f1)}")
    os.makedirs(pasta_faixa, exist_ok=True)

    df_faixa.to_csv(
        os.path.join(pasta_faixa, "metricas_faixa.csv"),
        index=False
    )

    return df_faixa


def executar_analise_todas_faixas():
    t0_total = time.time()

    print("====================================================")
    print("ANÁLISE DE FAIXAS COM AE + CALIBRAÇÃO SAUDÁVEL")
    print("====================================================")

    df = pd.read_pickle(ARQ_BASE).reset_index(drop=True)

    required_cols = {"temperatura_c", "falha"}
    missing = required_cols - set(df.columns)

    if len(missing) > 0:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

    faixas = gerar_faixas(df)

    print(f"Total de amostras: {len(df)}")
    print(f"Classes de dano: {sorted(df['falha'].unique())}")
    print(f"Faixas geradas: {faixas}")

    resultados_faixas = []

    for f0, f1 in faixas:
        t0 = time.time()

        df_faixa = analisar_faixa(df, f0, f1)

        if df_faixa is not None:
            resultados_faixas.append(df_faixa)

        print(f"Tempo da faixa {formatar_faixa(f0, f1)}: {time.time() - t0:.2f} s")

    if len(resultados_faixas) == 0:
        raise RuntimeError("Nenhuma faixa válida foi analisada.")

    df_long = pd.concat(resultados_faixas, axis=0, ignore_index=True)

    resumo = resumo_por_faixa(df_long)
    df_mono = checar_monotonicidade_por_faixa(df_long)
    df_rank = criar_ranking_faixas(df_long)

    df_long.to_csv(
        os.path.join(OUTPUT_DIR, "df_long_todas_faixas.csv"),
        index=False
    )

    resumo.to_csv(
        os.path.join(OUTPUT_DIR, "resumo_por_faixa_metodo_dano.csv"),
        index=False
    )

    df_mono.to_csv(
        os.path.join(OUTPUT_DIR, "monotonicidade_por_faixa.csv"),
        index=False
    )

    df_rank.to_csv(
        os.path.join(OUTPUT_DIR, "ranking_faixas.csv"),
        index=False
    )

    print("\n====================================================")
    print("RANKING DAS FAIXAS — AE CALIB SAUDÁVEL")
    print("====================================================")
    display(df_rank)

    if SALVAR_GRAFICOS_RESUMO:
        plot_heatmaps_comparacao(df_long)

    print("\n✅ Análise concluída.")
    print(f"Pasta de saída: {OUTPUT_DIR}")
    print(f"Tempo total: {time.time() - t0_total:.2f} s")

    return {
        "df_long": df_long,
        "resumo": resumo,
        "df_mono": df_mono,
        "df_rank": df_rank,
        "faixas": faixas
    }


# ============================================================
# 11) RODAR
# ============================================================

resultados = executar_analise_todas_faixas()

df_long = resultados["df_long"]
resumo = resultados["resumo"]
df_mono = resultados["df_mono"]
df_rank = resultados["df_rank"]
faixas = resultados["faixas"]

print("\nMelhores faixas pelo ranking:")
display(df_rank.head(10))
